# 第 7 章 ダミー変数: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/07-dummy-variables.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 80               # 標本の大きさ
interaction = True   # True: 女性ダミーと教育年数の交差項を入れる

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 男女の平均と、ダミー変数・交差項の回帰

In [ ]:
import numpy as np
import pandas as pd

from lib.chapter03.multiple import fit_ols
from lib.chapter07.dummies import draw_wage_sample, group_means, with_interaction

df = draw_wage_sample(n, np.random.default_rng(seed))
print("女性（1）と男性（0）の、対数賃金の平均")
print(group_means(df["lwage"], df["female"]).round(3).to_string())

x = df[["educ", "female"]].astype(float)
if interaction:
    x = x.join(with_interaction(x["educ"], x["female"]))
result = fit_ols(df["lwage"], x)
print(
    pd.DataFrame(
        {"係数": result.coefficients, "標準誤差": result.standard_errors}
    ).round(3)
)
print(f"R² {result.r_squared:.3f}")

## 図: 2 群のデータと回帰直線

In [ ]:
from lib.chapter07.plot import plot_group_lines

plot_group_lines(
    df["educ"].to_numpy(dtype=float),
    df["lwage"].to_numpy(),
    df["female"].to_numpy(dtype=float),
    interaction,
)

## 試してみよう

- `interaction` を `False` にすると、女性ダミーの係数は −0.010 になり、
  交差項を入れたときの −0.479 と大きく違います（交差項があると、女性ダミーは、
  教育年数が 0 のときの差を表します）
- `n` を 400 にすると、交差項の係数は 0.027（標準誤差 0.010）で、真の値（0.02）
  に近づきます。標本が大きいと、標準誤差が小さくなります